In [2]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [3]:
import os
from pathlib import Path
from functools import partial
from datetime import timedelta, datetime, date

import numpy as np
import polars as pl

from synth_data import create_daily_aggregation
from expressions.windows import agg_in_w_exprs

# Data catalog

In [4]:
data_path = Path(os.getenv("DATA_DIR", "/home/mle/data/"))
print(data_path.exists())
tmp_data_path = data_path / "tmp"
assets_data_path = data_path / "assets"

True


# Synthetic data

In [5]:
daily_agg_df = create_daily_aggregation(
    start=date(2026, 1, 1),
    end=date(2026, 4, 1),
    min_types_per_uid=2,
    max_types_per_uid=5,
)

In [6]:
daily_agg_df.head(5)

date,uid,val_type,val
date,i64,i8,i64
2026-01-01,0,1,7492
2026-01-01,0,9,427
2026-01-01,1,3,562
2026-01-01,1,8,9638
2026-01-01,2,2,9948


## Save synthetic data for loading the daily_agg_df from assets (the design data pipeline test task)

In [19]:
daily_agg_df.write_parquet(assets_data_path / "daily_agg.parquet", mkdir=True)

# Daily context

In Daily context features(from the expressions lib) computed only for one date called current_date

In [9]:
def daily_window_f(
    daily_agg_lf: pl.LazyFrame, current_date: date, period: int
) -> pl.LazyFrame:
    period_dt = timedelta(days=period)
    return (
        daily_agg_lf.filter(
            (current_date - period_dt < pl.col("date"))
            & (pl.col("date") <= current_date)
        )
        .group_by(["uid", "val_type"])
        .agg(agg_in_w_exprs(w=f"{period}"))
    )


def daily_multi_window_f(
    daily_agg_lf: pl.LazyFrame, current_date: date, periods: list[int]
) -> pl.DataFrame:
    daily_f_all: pl.DataFrame = pl.DataFrame()
    for period in periods:
        if daily_f_all.is_empty():
            daily_f_all = daily_window_f(
                daily_agg_lf=daily_agg_lf, current_date=current_date, period=period
            ).collect()
        else:
            daily_f_all = daily_f_all.join(
                daily_window_f(
                    daily_agg_lf=daily_agg_lf, current_date=current_date, period=period
                ).collect(),
                on=["uid", "val_type"],
                how="full",
                coalesce=True,
            )
    return daily_f_all

In [11]:
current_date = date(2025, 1, 24)
daily_features = daily_multi_window_f(
    daily_agg_lf=daily_agg_df.lazy(), current_date=current_date, periods=[3, 7]
)

In [12]:
daily_features.head()

uid,val_type,sum_val_by_type_in_3,min_val_by_type_in_3,max_val_by_type_in_3,sum_val_by_type_in_7,min_val_by_type_in_7,max_val_by_type_in_7
i64,i8,i64,i64,i64,i64,i64,i64
93,5,2472,2472,2472,11194,2472,8722
22,4,3840,3840,3840,9868,3840,6028
12,8,2647,2647,2647,4917,2270,2647
88,2,14256,1899,7797,14256,1899,7797
91,7,null,null,null,2852,270,2582


# Backfill context (task)
In Backfill context the features (expressions from the expressions lib) must be computed for each date in daily_agg_df dataframe

You need  to develop function for computing agg_in_w_exprs in Backfill context


# Backfill tests
You need to develop tests for Backfilling, tests must compare results of expression computing in Daily context with of Backfilling results

Think how we can use LazyFrame to optimize daily_multi_window_f?